<a href="https://colab.research.google.com/github/miraimeisatu/NEDO-2026/blob/main/%5BNEDO_1%5D%E3%82%A2%E3%83%89%E3%83%99%E3%83%B3%E3%83%88%E3%82%AB%E3%83%AC%E3%83%B3%E3%83%80_%E6%97%A5%E6%9C%AC%E5%85%A8%E5%9C%9F%E6%B5%81%E9%80%9A%E9%85%8D%E7%BD%AEv2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""
日本全土倉庫配置最適化プログラム v2.0

改善内容:
- Google Colab依存の排除
- エラーハンドリングの追加
- コードの関数化・モジュール化
- パラメータの定数化
- 結果の保存機能追加
- 可視化の強化
"""

import os
import sys
import json
import datetime
import time
from pathlib import Path

import pandas as pd
import numpy as np
import folium
from pulp import LpProblem, LpMinimize, LpVariable, lpSum, lpDot, LpStatus

# ============================================================================
# 定数定義
# ============================================================================

# 地球楕円体のパラメータ (km)
EQUATORIAL_RADIUS = 6378.140  # 赤道半径
POLAR_RADIUS = 6356.755       # 極半径

# 距離計算パラメータ
ROAD_DISTANCE_FACTOR = 1.3    # 直線距離から道路距離への変換係数

# 最適化パラメータ
DEFAULT_NUM_WAREHOUSES = 7    # 地方区分ごと
MAX_STORES_PER_WAREHOUSE = 500  # 倉庫あたりの最大店舗数
CAPACITY_MULTIPLIER = 4       # 容量計算の乗数

# 可視化パラメータ
MAP_INITIAL_LOCATION = [37.1039474,138.7660754]  # 地図の中心位置（東京駅）
MAP_INITIAL_ZOOM = 3.5
CIRCLE_MARKER_RADIUS = 5

# カラーパレット（倉庫ごとの色分け用）
COLOR_PALETTE = [
    '#FF6B6B',  # コーラルレッド
    '#4ECDC4',  # ターコイズ
    '#45B7D1',  # スカイブルー
    '#FFA07A',  # ライトサーモン
    '#98D8C8',  # ミントグリーン
    '#F7DC6F',  # サンフラワーイエロー
    '#BB8FCE',  # ラベンダー
    '#F8B88B',  # ピーチ
    '#85C1E2',  # パウダーブルー
    '#F1948A',  # ローズピンク
    '#7FB3D5',  # セレニティブルー
    '#FAD7A0'   # アプリコット
]
# ============================================================================
# ユーティリティ関数
# ============================================================================

def is_google_colab():
    """Google Colab環境かどうかを判定"""
    try:
        import google.colab
        return True
    except ImportError:
        return False


def load_csv_file(prompt_message, default_filename=None):
    """
    CSVファイルを読み込む（Google Colab / ローカル環境対応）

    Args:
        prompt_message: ファイル選択時のメッセージ
        default_filename: デフォルトのファイル名（ローカル環境用）

    Returns:
        pd.DataFrame: 読み込んだデータフレーム
    """
    if is_google_colab():
        from google.colab import files
        print(prompt_message)
        uploaded = files.upload()
        if not uploaded:
            raise ValueError("ファイルがアップロードされませんでした")
        filename = list(uploaded.keys())[0]
        return pd.read_csv(filename)
    else:
        if default_filename and os.path.exists(default_filename):
            print(f"{default_filename} を読み込みます")
            return pd.read_csv(default_filename)
        else:
            filename = input(f"{prompt_message}\nファイルパスを入力してください: ")
            if not os.path.exists(filename):
                raise FileNotFoundError(f"ファイルが見つかりません: {filename}")
            return pd.read_csv(filename)


def validate_dataframe(df, required_columns, df_name):
    """
    データフレームの必須カラムを検証

    Args:
        df: 検証するデータフレーム
        required_columns: 必須カラムのリスト
        df_name: データフレーム名（エラーメッセージ用）
    """
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(
            f"{df_name}に必須カラムがありません: {missing_columns}\n"
            f"存在するカラム: {list(df.columns)}"
        )
    print(f"✓ {df_name}の検証完了: {len(df)}行")


# ============================================================================
# 距離計算
# ============================================================================

def calculate_distance_lambert_andoyer(lon_a, lat_a, lon_b, lat_b):
    """
    Lambert-Andoyer法による地球楕円体面上の2点間の距離を計算

    Args:
        lon_a, lat_a: 地点Aの経度・緯度（度）
        lon_b, lat_b: 地点Bの経度・緯度（度）

    Returns:
        float: 距離（km）
    """
    F = (EQUATORIAL_RADIUS - POLAR_RADIUS) / EQUATORIAL_RADIUS

    rad_lat_a = np.radians(lat_a)
    rad_lon_a = np.radians(lon_a)
    rad_lat_b = np.radians(lat_b)
    rad_lon_b = np.radians(lon_b)

    pa = np.arctan(POLAR_RADIUS / EQUATORIAL_RADIUS * np.tan(rad_lat_a))
    pb = np.arctan(POLAR_RADIUS / EQUATORIAL_RADIUS * np.tan(rad_lat_b))

    cos_term = np.cos(pa) * np.cos(pb) * np.cos(rad_lon_a - rad_lon_b)
    xx = np.arccos(np.sin(pa) * np.sin(pb) + cos_term)

    # ゼロ除算を防ぐ
    if xx == 0:
        return 0.0

    sin_xx = np.sin(xx)
    cos_half = np.cos(xx / 2)
    sin_half = np.sin(xx / 2)

    if cos_half == 0 or sin_half == 0:
        return 0.0

    c1 = (sin_xx - xx) * (np.sin(pa) + np.sin(pb))**2 / cos_half**2
    c2 = (sin_xx + xx) * (np.sin(pa) - np.sin(pb))**2 / sin_half**2
    dr = F / 8 * (c1 - c2)
    dist = EQUATORIAL_RADIUS * (xx + dr)

    return 0.0 if np.isnan(dist) else dist


def create_distance_matrix(df_candidate, df_store):
    """
    倉庫候補地と店舗間の距離マトリクスを作成

    Args:
        df_candidate: 倉庫候補地データフレーム
        df_store: 店舗データフレーム

    Returns:
        dict: (候補地index, 店舗index) -> 道路距離(km) の辞書
    """
    print("\n距離マトリクスを計算中...")
    start_time = time.time()

    distance_dict = {}
    total_pairs = len(df_candidate) * len(df_store)

    for i, candidate in df_candidate.iterrows():
        for j, store in df_store.iterrows():
            direct_dist = calculate_distance_lambert_andoyer(
                candidate.lon, candidate.lat,
                store.lon, store.lat
            )
            # 直線距離を道路距離に変換
            road_dist = direct_dist * ROAD_DISTANCE_FACTOR
            distance_dict[(i, j)] = road_dist

    elapsed_time = time.time() - start_time
    print(f"✓ 距離計算完了: {total_pairs}ペア ({elapsed_time:.2f}秒)")

    # 統計情報を表示
    distances = list(distance_dict.values())
    print(f"  平均距離: {np.mean(distances):.2f} km")
    print(f"  最小距離: {np.min(distances):.2f} km")
    print(f"  最大距離: {np.max(distances):.2f} km")

    return distance_dict


# ============================================================================
# 最適化処理
# ============================================================================

def optimize_warehouse_location(
    df_candidate,
    df_store,
    distance_dict,
    num_warehouses=DEFAULT_NUM_WAREHOUSES
):
    """
    倉庫配置の最適化を実行

    目的関数: 総輸送コスト（距離×需要）の最小化

    制約条件:
    1. 建設する倉庫数は指定数
    2. 倉庫が稼働していない場合、その倉庫からの配送はゼロ
    3. 各倉庫の容量制約
    4. 各店舗は必ず1つの倉庫から配送を受ける
    5. フロー変数とマレータウスはバイナリ

    Args:
        df_candidate: 倉庫候補地データフレーム
        df_store: 店舗データフレーム
        distance_dict: 距離マトリクス
        num_warehouses: 建設する倉庫数

    Returns:
        tuple: (最適化問題オブジェクト, フロー変数配列, 倉庫変数リスト)
    """
    print(f"\n最適化を開始（倉庫数: {num_warehouses}箇所）...")
    start_time = time.time()

    # 問題を定義
    problem = LpProblem("warehouse_location_optimization", LpMinimize)

    # 決定変数を定義
    # flow[i][j]: 倉庫候補地iから店舗jへの配送の有無（0 or 1）
    var_flow = np.array([
        [LpVariable(f"flow_{i}_{j}", cat="Binary")
         for j in df_store.index]
        for i in df_candidate.index
    ])

    # wh[i]: 倉庫候補地iに倉庫を建設するか（0 or 1）
    var_wh = [
        LpVariable(f"warehouse_{i}", cat="Binary")
        for i in df_candidate.index
    ]

    # 目的関数: 総輸送コスト = Σ(配送フロー × 需要 × 距離)
    objective = lpSum([
        var_flow[i][j] * df_store.iloc[j]["demand"] * distance_dict[(i, j)]
        for i in df_candidate.index
        for j in df_store.index
    ])
    problem += objective

    # 制約1: 建設する倉庫数
    problem += lpSum(var_wh) == num_warehouses, "constraint_warehouse_count"

    # 制約2, 3: 倉庫の稼働条件と容量制約
    for i in df_candidate.index:
        # 倉庫が稼働していない場合、配送できない
        problem += (
            lpSum(var_flow[i]) <= MAX_STORES_PER_WAREHOUSE * var_wh[i],
            f"constraint_warehouse_operation_{i}"
        )

        # 倉庫の容量制約
        problem += (
            lpDot(var_flow[i], df_store["demand"]) <=
            df_candidate.iloc[i].capacity * CAPACITY_MULTIPLIER,
            f"constraint_warehouse_capacity_{i}"
        )

    # 制約4: 各店舗は必ず1つの倉庫から配送を受ける
    for j in df_store.index:
        problem += (
            lpSum(var_flow[:, j]) == 1,
            f"constraint_store_assignment_{j}"
        )

    # 最適化実行
    print("最適化計算中...")
    status = problem.solve()

    elapsed_time = time.time() - start_time
    print(f"✓ 最適化完了: {elapsed_time:.2f}秒")
    print(f"  ステータス: {LpStatus[status]}")

    if status != 1:  # 1 = Optimal
        print(f"⚠ 警告: 最適解が見つかりませんでした（ステータス: {LpStatus[status]}）")

    return problem, var_flow, var_wh


# ============================================================================
# 結果分析・出力
# ============================================================================

def analyze_optimization_results(
    problem,
    var_flow,
    var_wh,
    df_candidate,
    df_store,
    distance_dict
):
    """
    最適化結果を分析し、詳細情報を表示

    Args:
        problem: 最適化問題オブジェクト
        var_flow: フロー変数配列
        var_wh: 倉庫変数リスト
        df_candidate: 倉庫候補地データフレーム
        df_store: 店舗データフレーム
        distance_dict: 距離マトリクス

    Returns:
        pd.DataFrame: 結果を追加したdf_candidate
    """
    print("\n" + "="*70)
    print("最適化結果")
    print("="*70)

    # 選択された倉庫のリスト
    selected_warehouses = []
    warehouse_stats = []

    for i in df_candidate.index:
        if var_wh[i].value() == 1:
            selected_warehouses.append(i)

            # この倉庫が担当する店舗を特定
            assigned_stores = []
            total_demand = 0
            total_distance_demand = 0

            for j in df_store.index:
                if var_flow[i][j].value() == 1:
                    assigned_stores.append(j)
                    demand = df_store.iloc[j]["demand"]
                    distance = distance_dict[(i, j)]
                    total_demand += demand
                    total_distance_demand += demand * distance

            avg_distance = (total_distance_demand / total_demand
                          if total_demand > 0 else 0)

            warehouse_stats.append({
                'index': i,
                'name': df_candidate.iloc[i].source_name,
                'num_stores': len(assigned_stores),
                'total_demand': total_demand,
                'avg_distance': avg_distance,
                'utilization': total_demand / (df_candidate.iloc[i].capacity * CAPACITY_MULTIPLIER) * 100
            })

    # 結果を表示
    print(f"\n選択された倉庫: {len(selected_warehouses)}箇所")
    print("-" * 70)
    for stat in warehouse_stats:
        print(f"倉庫: {stat['name']}")
        print(f"  担当店舗数: {stat['num_stores']}店舗")
        print(f"  総需要: {stat['total_demand']:.0f}人")
        print(f"  平均配送距離: {stat['avg_distance']:.2f} km")
        print(f"  容量使用率: {stat['utilization']:.1f}%")
        print()

    # 全体統計
    total_cost = problem.objective.value()
    total_stores = len(df_store)
    total_demand = df_store["demand"].sum()
    avg_cost_per_store = total_cost / total_stores if total_stores > 0 else 0
    avg_cost_per_demand = total_cost / total_demand if total_demand > 0 else 0

    print("="*70)
    print(f"総輸送コスト: {total_cost:,.2f} km·人")
    print(f"店舗あたり平均コスト: {avg_cost_per_store:,.2f} km·人")
    print(f"需要あたり平均コスト: {avg_cost_per_demand:.2f} km")
    print("="*70)

    # df_candidateに結果を追加
    df_candidate_result = df_candidate.copy()
    df_candidate_result["selected"] = [
        1 if var_wh[i].value() == 1 else 0
        for i in df_candidate.index
    ]

    return df_candidate_result, warehouse_stats


def save_results(
    df_candidate_result,
    warehouse_stats,
    problem,
    output_dir="results"
):
    """
    結果をファイルに保存

    Args:
        df_candidate_result: 結果付き倉庫候補地データフレーム
        warehouse_stats: 倉庫統計情報
        problem: 最適化問題オブジェクト
        output_dir: 出力ディレクトリ
    """
    # 出力ディレクトリを作成
    Path(output_dir).mkdir(exist_ok=True)

    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

    # 1. 倉庫選択結果をCSVで保存
    result_csv = f"{output_dir}/warehouse_selection_{timestamp}.csv"
    df_candidate_result.to_csv(result_csv, index=False, encoding="utf-8-sig")
    print(f"\n✓ 倉庫選択結果を保存: {result_csv}")

    # 2. 統計情報をJSONで保存
    stats_json = f"{output_dir}/warehouse_stats_{timestamp}.json"
    stats_data = {
        "optimization_results": {
            "total_cost": problem.objective.value(),
            "timestamp": timestamp
        },
        "warehouses": warehouse_stats
    }
    with open(stats_json, "w", encoding="utf-8") as f:
        json.dump(stats_data, f, ensure_ascii=False, indent=2)
    print(f"✓ 統計情報を保存: {stats_json}")

    return result_csv, stats_json


# ============================================================================
# 地図可視化
# ============================================================================

def create_visualization_map(
    var_flow,
    var_wh,
    df_candidate,
    df_store,
    output_file="results/warehouse_map.html"
):
    """
    最適化結果を地図上に可視化

    Args:
        var_flow: フロー変数配列
        var_wh: 倉庫変数リスト
        df_candidate: 倉庫候補地データフレーム
        df_store: 店舗データフレーム
        output_file: 出力HTMLファイルパス

    Returns:
        folium.Map: 地図オブジェクト
    """
    print("\n地図を作成中...")

    # 地図オブジェクトを作成
    m = folium.Map(
        location=MAP_INITIAL_LOCATION,
        zoom_start=MAP_INITIAL_ZOOM,
        tiles='OpenStreetMap'
    )

    # 選択された倉庫を特定
    selected_warehouses = [
        i for i in df_candidate.index
        if var_wh[i].value() == 1
    ]

    # 倉庫マーカーを追加（大きな赤いマーカー）
    for i in selected_warehouses:
        warehouse = df_candidate.iloc[i]
        folium.Marker(
            location=[warehouse.lat, warehouse.lon],
            popup=f"<b>倉庫: {warehouse.source_name}</b>",
            tooltip=warehouse.source_name,
            icon=folium.Icon(color='red', icon='home', prefix='fa')
        ).add_to(m)

    # 店舗を倉庫ごとに色分けして表示
    for color_idx, warehouse_idx in enumerate(selected_warehouses):
        color = COLOR_PALETTE[color_idx % len(COLOR_PALETTE)]

        for j in df_store.index:
            if var_flow[warehouse_idx][j].value() == 1:
                store = df_store.iloc[j]

                # 需要に応じてマーカーサイズを変更
                radius = max(3, min(15, store.demand / 100))

                folium.CircleMarker(
                    location=[store.lat, store.lon],
                    radius=radius,
                    popup=f"店舗<br>需要: {store.demand}人",
                    color=color,
                    fill=True,
                    fillColor=color,
                    fillOpacity=0.6,
                    weight=2
                ).add_to(m)

    # 凡例を追加
    legend_html = '''
    <div style="position: fixed;
                bottom: 50px; right: 50px;
                background-color: white;
                border:2px solid grey;
                z-index:9999;
                font-size:14px;
                padding: 10px">
        <p><b>凡例</b></p>
        <p><i class="fa fa-home" style="color:red"></i> 倉庫</p>
        <p><i class="fa fa-circle" style="color:grey"></i> 店舗（色は担当倉庫）</p>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))

    # HTMLファイルとして保存
    Path(output_file).parent.mkdir(exist_ok=True)
    m.save(output_file)
    print(f"✓ 地図を保存: {output_file}")

    return m


# ============================================================================
# メイン処理
# ============================================================================

def main():
    """メイン処理"""
    print("="*70)
    print("日本全土倉庫配置最適化プログラム v2.0")
    print("="*70)

    try:
        # ステップ1: データ読み込み
        print("\n[ステップ1] データ読み込み")
        print("-" * 70)

        df_candidate = load_csv_file(
            "倉庫候補地のCSVファイルをアップロードしてください",
            default_filename="candidate.csv"
        )
        # Rename columns to match expected names
        if 'city' in df_candidate.columns and 'source_name' not in df_candidate.columns:
            df_candidate = df_candidate.rename(columns={'city': 'source_name'})
        if 'lng' in df_candidate.columns and 'lon' not in df_candidate.columns:
            df_candidate = df_candidate.rename(columns={'lng': 'lon'})

        validate_dataframe(
            df_candidate,
            ['source_name', 'lat', 'lon', 'capacity'],
            "倉庫候補地データ"
        )

        df_store = load_csv_file(
            "店舗のCSVファイルをアップロードしてください",
            default_filename="store.csv"
        )
        # Rename columns to match expected names for store data
        if 'lng' in df_store.columns and 'lon' not in df_store.columns:
            df_store = df_store.rename(columns={'lng': 'lon'})

        validate_dataframe(
            df_store,
            ['lat', 'lon', 'demand'],
            "店舗データ"
        )

        # ステップ2: 距離計算
        print("\n[ステップ2] 距離計算")
        print("-" * 70)
        distance_dict = create_distance_matrix(df_candidate, df_store)

        # ステップ3: 最適化実行
        print("\n[ステップ3] 最適化実行")
        print("-" * 70)
        problem, var_flow, var_wh = optimize_warehouse_location(
            df_candidate,
            df_store,
            distance_dict,
            num_warehouses=DEFAULT_NUM_WAREHOUSES
        )

        # ステップ4: 結果分析
        print("\n[ステップ4] 結果分析")
        print("-" * 70)
        df_candidate_result, warehouse_stats = analyze_optimization_results(
            problem,
            var_flow,
            var_wh,
            df_candidate,
            df_store,
            distance_dict
        )

        # ステップ5: 結果保存
        print("\n[ステップ5] 結果保存")
        print("-" * 70)
        save_results(df_candidate_result, warehouse_stats, problem)

        # ステップ6: 地図可視化
        print("\n[ステップ6] 地図可視化")
        print("-" * 70)
        map_obj = create_visualization_map(
            var_flow,
            var_wh,
            df_candidate,
            df_store
        )

        print("\n" + "="*70)
        print("すべての処理が完了しました！")
        print("="*70)

        # Google Colabの場合は地図を表示
        if is_google_colab():
            return map_obj

    except Exception as e:
        print(f"\n❌ エラーが発生しました: {e}")
        import traceback
        traceback.print_exc()
        sys.exit(1)


# ============================================================================
# 実行
# ============================================================================

if __name__ == "__main__":
    # パラメータのカスタマイズ例
    # DEFAULT_NUM_WAREHOUSES = 10  # 倉庫数を変更する場合

    result = main()

    # Google Colabの場合は地図を表示
    if is_google_colab() and result is not None:
        display(result)


日本全土倉庫配置最適化プログラム v2.0

[ステップ1] データ読み込み
----------------------------------------------------------------------
倉庫候補地のCSVファイルをアップロードしてください


Saving candidate.csv to candidate (1).csv
✓ 倉庫候補地データの検証完了: 64行
店舗のCSVファイルをアップロードしてください


Saving store.csv to store (1).csv
✓ 店舗データの検証完了: 631行

[ステップ2] 距離計算
----------------------------------------------------------------------

距離マトリクスを計算中...


/tmp/ipython-input-1959646802.py:149: RuntimeWarning: invalid value encountered in arccos
  xx = np.arccos(np.sin(pa) * np.sin(pb) + cos_term)


✓ 距離計算完了: 40384ペア (5.09秒)
  平均距離: 775.06 km
  最小距離: 0.00 km
  最大距離: 3735.41 km

[ステップ3] 最適化実行
----------------------------------------------------------------------

最適化を開始（倉庫数: 7箇所）...
最適化計算中...
✓ 最適化完了: 135.94秒
  ステータス: Optimal

[ステップ4] 結果分析
----------------------------------------------------------------------

最適化結果

選択された倉庫: 7箇所
----------------------------------------------------------------------
倉庫: 北海道札幌市 中央区
  担当店舗数: 70店舗
  総需要: 2277625人
  平均配送距離: 105.65 km
  容量使用率: 2.0%

倉庫: 山形県寒河江市
  担当店舗数: 77店舗
  総需要: 4295840人
  平均配送距離: 153.63 km
  容量使用率: 3.8%

倉庫: 東京都台東区
  担当店舗数: 154店舗
  総需要: 24698785人
  平均配送距離: 37.62 km
  容量使用率: 21.9%

倉庫: 愛知県蟹江町
  担当店舗数: 80店舗
  総需要: 5522513人
  平均配送距離: 88.47 km
  容量使用率: 4.9%

倉庫: 大阪府藤井寺市
  担当店舗数: 80店舗
  総需要: 6257064人
  平均配送距離: 45.55 km
  容量使用率: 5.6%

倉庫: 岡山県笠岡市
  担当店舗数: 69店舗
  総需要: 3609216人
  平均配送距離: 102.09 km
  容量使用率: 3.2%

倉庫: 熊本県南関町
  担当店舗数: 101店舗
  総需要: 4508899人
  平均配送距離: 188.26 km
  容量使用率: 4.0%

総輸送コスト: 3,820,776,826.91 km·人
店舗あたり平均コスト: 6,055,113.83